# PolypDB — YOLOv8n on WLI Modality
**Goal:** Train a lightweight polyp detector on White Light Imaging (WLI) data.
- Dataset: PolypDB (OSF: https://osf.io/pr7ms/)
- Model: YOLOv8n (nano) — small footprint for edge deployment
- GPU: Kaggle T4 (free tier)

**Expected output:**
- mAP@50 target: > 0.80
- Model size: < 10 MB (.pt), < 20 MB (.onnx)
- Inference: < 20 ms on T4

In [ ]:
# 1. Install dependencies
!pip install ultralytics==8.3.* -q
!pip install osfclient -q  # for OSF dataset download

In [ ]:
# 2. Download PolypDB from OSF
# OSF project: https://osf.io/pr7ms/
import subprocess, os

os.makedirs('/kaggle/working/data/polypdb', exist_ok=True)

# Download via osfclient (public project, no auth needed)
result = subprocess.run(
    ['osf', '-p', 'pr7ms', 'clone', '/kaggle/working/data/polypdb'],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if result.stdout else '')
print(result.stderr[-500:] if result.stderr else '')

In [ ]:
# 2b. Inspect downloaded structure
import os
for root, dirs, files in os.walk('/kaggle/working/data/polypdb'):
    depth = root.replace('/kaggle/working/data/polypdb', '').count(os.sep)
    if depth > 3:
        continue
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root)}/')
    if depth == 3:
        subindent = '  ' * (depth + 1)
        for f in files[:5]:
            print(f'{subindent}{f}')

In [ ]:
# 3. Convert COCO → YOLO format (WLI only)
import json
import shutil
from pathlib import Path

DATA_ROOT = Path('/kaggle/working/data/polypdb')
OUT_ROOT  = Path('/kaggle/working/data/polypdb_wli')
MODALITY  = 'WLI'

def coco_to_yolo_bbox(bbox, img_w, img_h):
    x, y, w, h = bbox
    cx = (x + w/2) / img_w
    cy = (y + h/2) / img_h
    return cx, cy, w/img_w, h/img_h

def convert_split(split):
    # Try common paths for coco JSON
    candidates = [
        DATA_ROOT / 'coco_labels' / f'{split}.json',
        DATA_ROOT / f'{split}.json',
        DATA_ROOT / 'annotations' / f'{split}.json',
    ]
    coco_json = next((p for p in candidates if p.exists()), None)
    if not coco_json:
        print(f'  WARNING: no COCO JSON found for split={split}')
        return 0

    with open(coco_json) as f:
        coco = json.load(f)

    ann_by_img = {}
    for ann in coco['annotations']:
        ann_by_img.setdefault(ann['image_id'], []).append(ann)

    img_dir  = OUT_ROOT / split / 'images'
    lbl_dir  = OUT_ROOT / split / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    kept = 0
    for img_info in coco['images']:
        fname = img_info['file_name']
        basename = Path(fname).name
        if not basename.upper().startswith(MODALITY):
            continue

        img_id = img_info['id']
        stem   = Path(fname).stem
        w, h   = img_info['width'], img_info['height']

        # Write label
        anns = ann_by_img.get(img_id, [])
        with open(lbl_dir / f'{stem}.txt', 'w') as lf:
            for ann in anns:
                if ann.get('bbox') and ann['bbox'][2] > 0 and ann['bbox'][3] > 0:
                    cx, cy, bw, bh = coco_to_yolo_bbox(ann['bbox'], w, h)
                    lf.write(f'0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n')

        # Find and copy image (search common locations)
        img_candidates = [
            DATA_ROOT / 'images' / basename,
            DATA_ROOT / 'images' / fname,
            DATA_ROOT / basename,
        ]
        src = next((p for p in img_candidates if p.exists()), None)
        if src:
            shutil.copy2(src, img_dir / basename)
        kept += 1

    print(f'  [{split}] {MODALITY}: kept {kept} images')
    return kept

for split in ['train', 'val', 'test']:
    convert_split(split)

print('\nConversion done.')

In [ ]:
# 4. Write dataset YAML
yaml_content = f"""path: {str(OUT_ROOT)}
train: train/images
val:   val/images
test:  test/images

nc: 1
names: ['polyp']
"""
yaml_path = Path('/kaggle/working/wli_yolov8n.yaml')
yaml_path.write_text(yaml_content)
print(yaml_content)

In [ ]:
# 5. Check GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# 6. Train YOLOv8n
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=str(yaml_path),
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    project='/kaggle/working/runs',
    name='yolov8n_wli',
    patience=20,
    save=True,
    plots=True,
    val=True,
    verbose=True,
)

In [ ]:
# 7. Evaluate — mAP, speed, size
import time, os
import numpy as np

best_pt = '/kaggle/working/runs/yolov8n_wli/weights/best.pt'
model = YOLO(best_pt)

# Validation metrics
metrics = model.val(data=str(yaml_path), device=0)
map50    = metrics.box.map50
map50_95 = metrics.box.map

# Inference speed on GPU (T4)
dummy = torch.zeros(1, 3, 640, 640).cuda()
for _ in range(5):  # warmup
    model.predict(source=dummy, verbose=False, device=0)
times = []
for _ in range(50):
    t0 = time.perf_counter()
    model.predict(source=dummy, verbose=False, device=0)
    times.append((time.perf_counter() - t0) * 1000)
avg_ms_gpu = np.mean(times)

# Model sizes
pt_mb = os.path.getsize(best_pt) / 1e6

# ONNX export
model.export(format='onnx', imgsz=640)
onnx_path = best_pt.replace('.pt', '.onnx')
onnx_mb = os.path.getsize(onnx_path) / 1e6 if os.path.exists(onnx_path) else None

print('='*55)
print('RESULTS — PolypDB YOLOv8n WLI')
print('='*55)
print(f'mAP@50:           {map50:.4f}  ({map50*100:.1f}%)')
print(f'mAP@50-95:        {map50_95:.4f}  ({map50_95*100:.1f}%)')
print(f'Inference (T4):   {avg_ms_gpu:.1f} ms  ({1000/avg_ms_gpu:.0f} FPS)')
print(f'Model .pt size:   {pt_mb:.1f} MB')
if onnx_mb:
    print(f'Model .onnx size: {onnx_mb:.1f} MB')
print('='*55)

In [ ]:
# 8. Visualize predictions on test set (sample)
from ultralytics import YOLO
from pathlib import Path
import random

test_images = list((OUT_ROOT / 'test' / 'images').glob('*.jpg')) + \
              list((OUT_ROOT / 'test' / 'images').glob('*.png'))
sample = random.sample(test_images, min(8, len(test_images)))

model = YOLO(best_pt)
results = model.predict(source=sample, save=True, project='/kaggle/working/runs', name='viz_wli', conf=0.25)

# Display in notebook
from IPython.display import Image, display
viz_dir = Path('/kaggle/working/runs/viz_wli')
for img_path in sorted(viz_dir.glob('*.jpg'))[:4]:
    display(Image(filename=str(img_path), width=480))